In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm


In [ ]:
# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

df = pd.read_csv(f"{DATA_PATH}/all_data.csv", sep="\t")

In [ ]:
cols_to_use = ["Sex", "Age", "Tumor", "CC", "PreOP CTx", "Thermoablation", "sPCI", "pPCI"]
df = df[cols_to_use].copy()

In [ ]:
# data wrangling
df['Sex'] = df['Sex']-1
df['PreOP CTx'] = df['PreOP CTx'].apply(lambda x: 1 if x >= 1 else x)

keep_tumor = [1, 5, 4, 6, 7, 3, 2]  # remove 8 if you decide to drop it

# Filter rows
df = df[df["Tumor"].isin(keep_tumor)].copy()
df['Tumor'] = df['Tumor'].apply(lambda x: f"type_{x}")

df = df[(df['CC']==0) | (df['CC']==1)].reset_index(drop=True)  # keep only CC0 and CC1

In [ ]:
# --- 1. Rename columns ---
df = df.rename(columns={
    "PreOP CTx" : "PreOP_CTx",
})

In [ ]:
# --- 5. One-hot encode Tumor (most frequent as reference) ---
# Tumor "1" (n=122) is the natural reference category
df = pd.get_dummies(df, columns=["Tumor"], drop_first=False, dtype=int)
df = df.drop(columns=["Tumor_type_1"], inplace=False)  # explicitly set Tumor_1 as reference

# --- 6. Create outcome variables ---
df["raw_diff"] = df["sPCI"] - df["pPCI"]
df["abs_diff"] = np.abs(df["raw_diff"])


In [ ]:
feature_cols = [
    'Sex', 'Age', 'CC', 'PreOP_CTx', 'Thermoablation', 'Tumor_type_2', 'Tumor_type_3', 'Tumor_type_4', 'Tumor_type_5',
       'Tumor_type_6', 'Tumor_type_7'
]

X     = df[feature_cols]
y_raw = df["raw_diff"]
y_abs = df["abs_diff"]

print(f"Final feature matrix: {X.shape}")

In [ ]:
X = sm.add_constant(X)

## Quantile Regression at the Median (q = 0.5)

Models the **median** of `raw_diff` — no distributional assumption on residuals.

In [ ]:
import statsmodels.formula.api as smf

qr50 = smf.quantreg(
    "raw_diff ~ Sex + Age + CC + PreOP_CTx + Thermoablation + "
    "Tumor_type_2 + Tumor_type_3 + Tumor_type_4 + Tumor_type_5 + Tumor_type_6 + Tumor_type_7",
    data=df
).fit(q=0.5)

print(qr50.summary())

## Quantile Process: q = 0.25, 0.50, 0.75, 0.90

Fit the model across multiple quantiles to see how effects change across the distribution of `raw_diff`.

In [ ]:
formula = (
    "raw_diff ~ Sex + Age + CC + PreOP_CTx + Thermoablation + "
    "Tumor_type_2 + Tumor_type_3 + Tumor_type_4 + Tumor_type_5 + Tumor_type_6 + Tumor_type_7"
)

quantiles = [0.25, 0.50, 0.75, 0.90]
qr_models = {q: smf.quantreg(formula, data=df).fit(q=q) for q in quantiles}

# Summary table: coefficients and p-values across quantiles
predictors = qr_models[0.5].params.index.tolist()

print(f"{'Predictor':<22}", end="")
for q in quantiles:
    print(f"  {'q='+str(q):<20}", end="")
print()
print("-" * (22 + 22 * len(quantiles)))

for pred in predictors:
    print(f"{pred:<22}", end="")
    for q in quantiles:
        m   = qr_models[q]
        coef = m.params[pred]
        pval = m.pvalues[pred]
        sig  = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else ""))
        print(f"  {coef:+.3f} p={pval:.3f} {sig:<4}", end="")
    print()

## Coefficient Plot across Quantiles

Visualise how each predictor's effect evolves from lower to upper quantiles, with 95% confidence bands.

In [ ]:
plot_quantiles = np.arange(0.1, 0.96, 0.05)
plot_models    = {q: smf.quantreg(formula, data=df).fit(q=q) for q in plot_quantiles}

# OLS reference line (with HC3 SEs)
ols_model  = smf.ols(formula, data=df).fit()
ols_robust = ols_model.get_robustcov_results(cov_type="HC3")

# get_robustcov_results returns ndarray params/conf_int, so reattach predictor names
ols_params = pd.Series(ols_robust.params, index=ols_model.params.index)
ols_ci = pd.DataFrame(ols_robust.conf_int(), index=ols_model.params.index, columns=[0, 1])

plot_predictors = [p for p in predictors if p != "Intercept"]
ncols = 3
nrows = -(-len(plot_predictors) // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, pred in zip(axes, plot_predictors):
    coefs = [plot_models[q].params[pred]  for q in plot_quantiles]
    ci_lo = [plot_models[q].conf_int().loc[pred, 0] for q in plot_quantiles]
    ci_hi = [plot_models[q].conf_int().loc[pred, 1] for q in plot_quantiles]

    ax.fill_between(plot_quantiles, ci_lo, ci_hi, alpha=0.2, color="steelblue")
    ax.plot(plot_quantiles, coefs, color="steelblue", lw=2, label="QR coef")
    # OLS reference
    ols_coef = ols_params[pred]
    ols_lo   = ols_ci.loc[pred, 0]
    ols_hi   = ols_ci.loc[pred, 1]
    ax.axhline(ols_coef, color="red",  linestyle="-",  lw=1.5, label="OLS (HC3)")
    ax.axhspan(ols_lo,   ols_hi,       alpha=0.15,     color="red")

    ax.set_title(pred)
    ax.set_xlabel("Quantile")
    ax.set_ylabel("Coefficient")

# hide unused axes
for ax in axes[len(plot_predictors):]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", fontsize=10)
plt.suptitle("Quantile regression coefficients vs OLS (HC3)", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()


## Pseudo-R² across Quantiles

The quantile regression pseudo-R² (Koenker & Machado, 1999) measures goodness of fit at each quantile — analogous to R² but for the quantile loss function.

In [ ]:
pseudo_r2 = [plot_models[q].prsquared for q in plot_quantiles]

plt.figure(figsize=(7, 4))
plt.plot(plot_quantiles, pseudo_r2, color="steelblue", lw=2, marker="o", markersize=4)
plt.axhline(ols_model.rsquared, color="red", linestyle="--", label=f"OLS R² = {ols_model.rsquared:.3f}")
plt.xlabel("Quantile")
plt.ylabel("Pseudo-R²")
plt.title("Quantile regression pseudo-R² (Koenker–Machado) vs OLS R²")
plt.legend()
plt.tight_layout()
plt.show()

print("\nPseudo-R² at selected quantiles:")
for q in [0.25, 0.50, 0.75, 0.90]:
    print(f"  q={q:.2f}  pseudo-R² = {qr_models[q].prsquared:.4f}")

## Diagnostics

### Diag 1 — Residual Sign Proportions

At quantile *q*, exactly *q × 100%* of residuals should be negative (this is the defining property of quantile regression). Deviation indicates numerical convergence issues.

In [ ]:
print(f"{'Quantile':>10}  {'Expected %neg':>14}  {'Actual %neg':>12}  {'N neg':>6}  {'OK?'}")
print("-" * 55)
for q in quantiles:
    resid_q = qr_models[q].resid
    pct_neg  = (resid_q < 0).mean()
    ok       = "✓" if abs(pct_neg - q) < 0.05 else "✗"
    print(f"  q={q:.2f}    {q*100:>10.1f}%    {pct_neg*100:>10.1f}%    {(resid_q<0).sum():>5}    {ok}")

### Diag 2 — Residuals vs Fitted

Check for systematic patterns or heteroskedasticity in residuals at each quantile.

In [ ]:
fig, axes = plt.subplots(1, len(quantiles), figsize=(5 * len(quantiles), 4), sharey=False)

for ax, q in zip(axes, quantiles):
    m      = qr_models[q]
    resid  = m.resid
    fitted = m.fittedvalues
    ax.scatter(fitted, resid, alpha=0.4, s=15, color="steelblue")
    ax.axhline(0, color="black", linestyle="--", lw=0.8)
    ax.set_xlabel("Fitted values")
    ax.set_ylabel("Residuals")
    ax.set_title(f"q = {q}")

plt.suptitle("Residuals vs Fitted — Quantile Regression", y=1.02)
plt.tight_layout()
plt.show()

### Diag 3 — Residuals vs Predictors (Spearman)

Spearman correlations between residuals and each predictor should be ~0. Any significant correlation suggests a missed non-linear term or an omitted variable.

In [ ]:
from scipy import stats

print(f"{'Predictor':<22}", end="")
for q in quantiles:
    print(f"  {'q='+str(q):<18}", end="")
print()
print("-" * (22 + 20 * len(quantiles)))

for col in feature_cols:
    print(f"{col:<22}", end="")
    for q in quantiles:
        resid_q = qr_models[q].resid
        r, p    = stats.spearmanr(df[col], resid_q)
        sig     = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
        print(f"  rho={r:+.2f} p={p:.3f}{sig:<4}", end="")
    print()

### Diag 4 — Bootstrap CIs vs Asymptotic SEs (median model)

The default SEs in `quantreg` use a kernel-based sparsity estimator whose bandwidth choice can affect results. Bootstrap CIs are a distribution-free alternative. Disagreement between the two flags predictors where the sparsity estimator is unreliable.

In [ ]:
np.random.seed(42)
n_boot = 2000
n      = len(df)
param_names = qr50.params.index.tolist()
boot_params = np.zeros((n_boot, len(param_names)))

for i in range(n_boot):
    idx   = np.random.choice(n, size=n, replace=True)
    b     = smf.quantreg(formula, data=df.iloc[idx]).fit(q=0.5, disp=False)
    boot_params[i] = b.params.values

boot_se = boot_params.std(axis=0)
ci_lo   = np.percentile(boot_params, 2.5,  axis=0)
ci_hi   = np.percentile(boot_params, 97.5, axis=0)

print(f"{'Predictor':<22} {'coef':>7}  {'asym_SE':>8}  {'boot_SE':>8}  {'95% CI (bootstrap)'}")
print("-" * 75)
for name, coef, ase, bse, lo, hi in zip(
        param_names, qr50.params, qr50.bse, boot_se, ci_lo, ci_hi):
    sig = "*" if lo > 0 or hi < 0 else ""
    print(f"{name:<22} {coef:+7.3f}  {ase:8.3f}  {bse:8.3f}  [{lo:+.3f}, {hi:+.3f}] {sig}")

### Diag 5 — Symmetry Test (q = 0.25 vs q = 0.75)

Tests whether the effect of each predictor at the lower and upper quartile are equal. A significant difference means the effect is **asymmetric** — larger in one tail than the other. Uses a Wald-type test based on bootstrap samples.

In [ ]:
np.random.seed(0)
n_boot2 = 2000
boot_25  = np.zeros((n_boot2, len(param_names)))
boot_75  = np.zeros((n_boot2, len(param_names)))

for i in range(n_boot2):
    idx = np.random.choice(n, size=n, replace=True)
    sub = df.iloc[idx]
    boot_25[i] = smf.quantreg(formula, data=sub).fit(q=0.25, disp=False).params.values
    boot_75[i] = smf.quantreg(formula, data=sub).fit(q=0.75, disp=False).params.values

diff        = boot_75 - boot_25
diff_mean   = diff.mean(axis=0)
diff_se     = diff.std(axis=0)
diff_ci_lo  = np.percentile(diff, 2.5,  axis=0)
diff_ci_hi  = np.percentile(diff, 97.5, axis=0)

print("Symmetry test: effect at q=0.75 minus effect at q=0.25")
print("A CI excluding zero means the effect differs between tails.\n")
print(f"{'Predictor':<22} {'Δcoef':>7}  {'boot_SE':>8}  {'95% CI of Δ':<24}  {'Asymmetric?'}")
print("-" * 75)
for pname, dm, ds, lo, hi in zip(param_names, diff_mean, diff_se, diff_ci_lo, diff_ci_hi):
    asym = "YES *" if lo > 0 or hi < 0 else ""
    print(f"{pname:<22} {dm:+7.3f}  {ds:8.3f}  [{lo:+.3f}, {hi:+.3f}]          {asym}")

### Diag 6 — Quantile Crossing

Checks whether predicted quantiles are properly ordered (q25 ≤ q50 ≤ q75 ≤ q90). Crossing indicates model instability for individual observations.

In [ ]:
# Build prediction matrix from fitted models
preds_df = pd.DataFrame(index=df.index)
for q in quantiles:
    preds_df[f"q{int(q*100)}"] = qr_models[q].predict(df)

# Pairwise crossing checks
cross_25_50 = preds_df["q25"] > preds_df["q50"]
cross_50_75 = preds_df["q50"] > preds_df["q75"]
cross_75_90 = preds_df["q75"] > preds_df["q90"]

any_crossing_full = cross_25_50 | cross_50_75 | cross_75_90

# Report
print("\nQuantile Crossing Diagnostics (full model)")
print("-" * 40)
print(f"q25 > q50: {cross_25_50.mean():.3%}")
print(f"q50 > q75: {cross_50_75.mean():.3%}")
print(f"q75 > q90: {cross_75_90.mean():.3%}")
print(f"ANY crossing: {any_crossing_full.mean():.3%}")


# Reduced Model — Age Excluded

Age was not significant at any quantile (all p > 0.10). The following cells repeat the full analysis without Age as a predictor.

In [ ]:
r_feature_cols = [
    'Sex', 'CC', 'PreOP_CTx', 'Thermoablation',
    'Tumor_type_2', 'Tumor_type_3', 'Tumor_type_4',
    'Tumor_type_5', 'Tumor_type_6', 'Tumor_type_7'
]

r_formula = (
    "raw_diff ~ Sex + CC + PreOP_CTx + Thermoablation + "
    "Tumor_type_2 + Tumor_type_3 + Tumor_type_4 + Tumor_type_5 + Tumor_type_6 + Tumor_type_7"
)

print(f"Predictors: {r_feature_cols}")

## Quantile Regression at the Median (q = 0.5)

In [ ]:
r_qr50 = smf.quantreg(r_formula, data=df).fit(q=0.5)
print(r_qr50.summary())

## Quantile Process: q = 0.25, 0.50, 0.75, 0.90

In [ ]:
r_quantiles = [0.25, 0.50, 0.75, 0.90]
r_preds = pd.DataFrame(index=df.index)
r_qr_models = {q: smf.quantreg(r_formula, data=df).fit(q=q) for q in r_quantiles}

r_predictors = r_qr_models[0.5].params.index.tolist()

print(f"{'Predictor':<22}", end="")
for q in r_quantiles:
    print(f"  {'q='+str(q):<20}", end="")
print()
print("-" * (22 + 22 * len(r_quantiles)))

for pred in r_predictors:
    print(f"{pred:<22}", end="")
    for q in r_quantiles:
        m    = r_qr_models[q]
        coef = m.params[pred]
        pval = m.pvalues[pred]
        sig  = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else ""))
        print(f"  {coef:+.3f} p={pval:.3f} {sig:<4}", end="")
    print()

## Coefficient Plot across Quantiles

In [ ]:
r_plot_quantiles = np.arange(0.1, 0.96, 0.05)
r_plot_models    = {q: smf.quantreg(r_formula, data=df).fit(q=q) for q in r_plot_quantiles}

r_ols_model  = smf.ols(r_formula, data=df).fit()
r_ols_robust = r_ols_model.get_robustcov_results(cov_type="HC3")
r_ols_params = pd.Series(r_ols_robust.params, index=r_ols_model.params.index)
r_ols_ci     = pd.DataFrame(r_ols_robust.conf_int(), index=r_ols_model.params.index, columns=[0, 1])

r_plot_predictors = [p for p in r_predictors if p != "Intercept"]
ncols = 3
nrows = -(-len(r_plot_predictors) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, pred in zip(axes, r_plot_predictors):
    coefs = [r_plot_models[q].params[pred] for q in r_plot_quantiles]
    ci_lo = [r_plot_models[q].conf_int().loc[pred, 0] for q in r_plot_quantiles]
    ci_hi = [r_plot_models[q].conf_int().loc[pred, 1] for q in r_plot_quantiles]

    ax.fill_between(r_plot_quantiles, ci_lo, ci_hi, alpha=0.2, color="steelblue")
    ax.plot(r_plot_quantiles, coefs, color="steelblue", lw=2, label="QR coef")

    ols_coef = r_ols_params[pred]
    ols_lo   = r_ols_ci.loc[pred, 0]
    ols_hi   = r_ols_ci.loc[pred, 1]
    ax.axhline(ols_coef, color="red", linestyle="-", lw=1.5, label="OLS (HC3)")
    ax.axhspan(ols_lo, ols_hi, alpha=0.15, color="red")

    ax.set_title(pred)
    ax.set_xlabel("Quantile")
    ax.set_ylabel("Coefficient")

for ax in axes[len(r_plot_predictors):]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", fontsize=10)
plt.suptitle("Reduced model (no Age): QR coefficients vs OLS (HC3)", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## Pseudo-R² across Quantiles

In [ ]:
r_pseudo_r2 = [r_plot_models[q].prsquared for q in r_plot_quantiles]

plt.figure(figsize=(7, 4))
plt.plot(r_plot_quantiles, r_pseudo_r2, color="steelblue", lw=2, marker="o", markersize=4)
plt.axhline(r_ols_model.rsquared, color="red", linestyle="--",
            label=f"OLS R² = {r_ols_model.rsquared:.3f}")
plt.xlabel("Quantile")
plt.ylabel("Pseudo-R²")
plt.title("Reduced model (no Age): pseudo-R² vs OLS R²")
plt.legend()
plt.tight_layout()
plt.show()

print("\nPseudo-R² at selected quantiles:")
for q in [0.25, 0.50, 0.75, 0.90]:
    print(f"  q={q:.2f}  pseudo-R² = {r_qr_models[q].prsquared:.4f}")

## Diagnostics

### Diag 1 — Residual Sign Proportions

In [ ]:
print(f"{'Quantile':>10}  {'Expected %neg':>14}  {'Actual %neg':>12}  {'N neg':>6}  {'OK?'}")
print("-" * 55)
for q in r_quantiles:
    resid_q = r_qr_models[q].resid
    pct_neg = (resid_q < 0).mean()
    ok      = "✓" if abs(pct_neg - q) < 0.05 else "✗"
    print(f"  q={q:.2f}    {q*100:>10.1f}%    {pct_neg*100:>10.1f}%    {(resid_q<0).sum():>5}    {ok}")

### Diag 2 — Residuals vs Fitted

In [ ]:
fig, axes = plt.subplots(1, len(r_quantiles), figsize=(5 * len(r_quantiles), 4), sharey=False)

for ax, q in zip(axes, r_quantiles):
    m      = r_qr_models[q]
    resid  = m.resid
    fitted = m.fittedvalues
    ax.scatter(fitted, resid, alpha=0.4, s=15, color="steelblue")
    ax.axhline(0, color="black", linestyle="--", lw=0.8)
    ax.set_xlabel("Fitted values")
    ax.set_ylabel("Residuals")
    ax.set_title(f"q = {q}")

plt.suptitle("Residuals vs Fitted — Reduced model (no Age)", y=1.02)
plt.tight_layout()
plt.show()

### Diag 3 — Residuals vs Predictors (Spearman)

In [ ]:
from scipy import stats

print(f"{'Predictor':<22}", end="")
for q in r_quantiles:
    print(f"  {'q='+str(q):<18}", end="")
print()
print("-" * (22 + 20 * len(r_quantiles)))

for col in r_feature_cols:
    print(f"{col:<22}", end="")
    for q in r_quantiles:
        resid_q = r_qr_models[q].resid
        r, p    = stats.spearmanr(df[col], resid_q)
        sig     = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
        print(f"  rho={r:+.2f} p={p:.3f}{sig:<4}", end="")
    print()

### Diag 4 — Bootstrap CIs vs Asymptotic SEs (median model)

In [ ]:
np.random.seed(42)
r_n_boot    = 2000
r_n         = len(df)
r_param_names = r_qr50.params.index.tolist()
r_boot_params = np.zeros((r_n_boot, len(r_param_names)))

for i in range(r_n_boot):
    idx = np.random.choice(r_n, size=r_n, replace=True)
    b   = smf.quantreg(r_formula, data=df.iloc[idx]).fit(q=0.5, disp=False)
    r_boot_params[i] = b.params.values

r_boot_se = r_boot_params.std(axis=0)
r_ci_lo   = np.percentile(r_boot_params, 2.5,  axis=0)
r_ci_hi   = np.percentile(r_boot_params, 97.5, axis=0)

print(f"{'Predictor':<22} {'coef':>7}  {'asym_SE':>8}  {'boot_SE':>8}  {'95% CI (bootstrap)'}")
print("-" * 75)
for name, coef, ase, bse, lo, hi in zip(
        r_param_names, r_qr50.params, r_qr50.bse, r_boot_se, r_ci_lo, r_ci_hi):
    sig = "*" if lo > 0 or hi < 0 else ""
    print(f"{name:<22} {coef:+7.3f}  {ase:8.3f}  {bse:8.3f}  [{lo:+.3f}, {hi:+.3f}] {sig}")

### Diag 5 — Symmetry Test (q = 0.25 vs q = 0.75)

In [ ]:
np.random.seed(0)
r_n_boot2  = 2000
r_boot_25  = np.zeros((r_n_boot2, len(r_param_names)))
r_boot_75  = np.zeros((r_n_boot2, len(r_param_names)))

for i in range(r_n_boot2):
    idx = np.random.choice(r_n, size=r_n, replace=True)
    sub = df.iloc[idx]
    r_boot_25[i] = smf.quantreg(r_formula, data=sub).fit(q=0.25, disp=False).params.values
    r_boot_75[i] = smf.quantreg(r_formula, data=sub).fit(q=0.75, disp=False).params.values

r_diff        = r_boot_75 - r_boot_25
r_diff_mean   = r_diff.mean(axis=0)
r_diff_se     = r_diff.std(axis=0)
r_diff_ci_lo  = np.percentile(r_diff, 2.5,  axis=0)
r_diff_ci_hi  = np.percentile(r_diff, 97.5, axis=0)

print("Symmetry test: effect at q=0.75 minus effect at q=0.25")
print("A CI excluding zero means the effect differs between tails.\n")
print(f"{'Predictor':<22} {'Δcoef':>7}  {'boot_SE':>8}  {'95% CI of Δ':<24}  {'Asymmetric?'}")
print("-" * 75)
for pname, dm, ds, lo, hi in zip(r_param_names, r_diff_mean, r_diff_se, r_diff_ci_lo, r_diff_ci_hi):
    asym = "YES *" if lo > 0 or hi < 0 else ""
    print(f"{pname:<22} {dm:+7.3f}  {ds:8.3f}  [{lo:+.3f}, {hi:+.3f}]          {asym}")

## Quantile Crossing

In [ ]:
# Build prediction matrix from your fitted models
for q in r_quantiles:
    r_preds[f"q{int(q*100)}"] = r_qr_models[q].predict(df)

# Pairwise crossing checks
cross_25_50 = r_preds["q25"] > r_preds["q50"]
cross_50_75 = r_preds["q50"] > r_preds["q75"]
cross_75_90 = r_preds["q75"] > r_preds["q90"]

# Aggregate crossing
any_crossing = cross_25_50 | cross_50_75 | cross_75_90

# Report
print("\nQuantile Crossing Diagnostics")
print("-" * 40)
print(f"q25 > q50: {cross_25_50.mean():.3%}")
print(f"q50 > q75: {cross_50_75.mean():.3%}")
print(f"q75 > q90: {cross_75_90.mean():.3%}")
print(f"ANY crossing: {any_crossing.mean():.3%}")